In [34]:
import os
import glob
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader

from dotenv import load_dotenv

In [35]:
MODEL = "gpt-4.1-mini"
DB_NAME = str(Path.cwd().parent / "vector_db")
KNOWLEDGE_BASE = str(Path.cwd().parent / "CMS Manuals")

load_dotenv(override=True)

open_ai_key = os.getenv('OPENAI_API_KEY')
if open_ai_key:
    print('OpenAI API key found')
else:
    print('OpenAI key not found')

OpenAI API key found


In [36]:
embeddings_openAI = OpenAIEmbeddings(model="text-embedding-3-large")
embeddings_HF = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


In [37]:
def fetch_documents():
    documents = []
    for pdf in Path('../CMS Manuals').glob('*.pdf'):
        print('PDF Found')
        loader = PyPDFLoader(str(pdf))
        docs = loader.load()
        documents.extend(docs)    
    print(f'{len(documents)} documents loaded')
    return documents

In [38]:
def format_sources(docs):
    sources = []

    for doc in docs:
        source = doc.metadata.get('source', 'Unknown Source')
        page_number = doc.metadata.get('page', None)

        if page_number is not None:
            page_number += 1           # Page numbers start at 0

        sources.append(f'- {source}, Page {page_number}')
    
    return "\n".join(sorted(set(sources)))

In [39]:
def create_chunks(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=500)
    chunks = text_splitter.split_documents(documents)
    return chunks

In [40]:
def create_embeddings(chunks):
    if os.path.exists(DB_NAME):     # If vectorstore exists, delete it. Allows for rerunning
        Chroma(persist_directory=DB_NAME, embedding_function=embeddings_openAI).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings_openAI, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
    dimensions = len(sample_embedding)
    print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore

In [41]:
import re

def fix_spaced_text(text):
    # joins single letters separated by spaces
    text = re.sub(r'(?<=\b\w) (?=\w\b)', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [42]:
def ingestion():
    docs = fetch_documents()
    
    for doc in docs:
        doc.page_content = fix_spaced_text(doc.page_content)
    
    chunks = create_chunks(docs)
    create_embeddings(chunks)
    print(f'Ingestion Complete. {len(docs)} documents loaded.')

In [43]:
ingestion()

PDF Found
PDF Found
PDF Found
PDF Found


Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)


PDF Found
PDF Found
PDF Found
PDF Found
332 documents loaded
There are 719 vectors with 3,072 dimensions in the vector store
Ingestion Complete. 332 documents loaded.


In [44]:
# embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
RETRIEVAL_K = 10

In [45]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage, convert_to_messages
from langchain_core.documents import Document

In [46]:
SYSTEM_PROMPT = """
You are a CMS manual question-answering assistant.

Answer the user's question using the context below.

Important rules:
- If the context contains relevant information, give the best supported answer.
- You may infer a direct answer from the context.
- Do not say the context is missing if the context discusses the same rule using different wording.
- If the context truly has no relevant information, say: "I could not find this in the provided CMS manuals."

Context:
{context}
"""

In [47]:
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings_openAI)
retriever = vectorstore.as_retriever(search_kwargs={'k' : RETRIEVAL_K})
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [48]:
def fetch_context(question: str):
    return retriever.invoke(question)

In [49]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import gradio as gr

def combined_question(question: str, history: list = None) -> str:
    history = history or []

    prior = "\n".join(user_msg for user_msg, assistant_msg in history)
    return prior + "\n" + question


def convert_to_messages(history: list = None):
    history = history or []

    messages = []
    for user_msg, assistant_msg in history:
        messages.append(HumanMessage(content=user_msg))
        messages.append(AIMessage(content=assistant_msg))

    return messages


def answer_question(question: str, history: list = None) -> str:
    history = history or []

    combined = combined_question(question, history)
    
    rewrite_prompt = f"""
    Rewrite this question for document retrieval.
    Question: {combined}
    """
    
    docs = fetch_context(rewrite_prompt)

    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)

    messages = [SystemMessage(content=system_prompt)]
    messages.extend(convert_to_messages(history))
    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)
    #print('CONTEXT SENT TO LLM')
    #print(context[:3000])
    sources = format_sources(docs)

    return f'{response.content}\nSources: {sources}'


gr.ChatInterface(answer_question).launch()

/Users/sumairparacha/projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [50]:
# question = "Can Medicare information be disclosed to a spouse without the beneficiary's consent?"

# docs = fetch_context(question)

# for i, doc in enumerate(docs):
#     print(f"\nDOC {i}")
#     print(doc.page_content[:1000])


DOC 0
of a beneficiary to the extent necessary to protect the beneficiary's rights under title II or XVIII. L. Disclosure without Consent Below are listed the situations in which data on identifiable individuals may be released without the individual's consent. The disclosure is permitted if the disclosure would be: • To DHHS employees and officers who need the records to perform their duties; • Required by the FOIA; • To the Bureau of Census; • For research purposes under certain circumstances; • To the National Archives; • For law enforcement activities if the activity is authorized by law and the request is from the head of the agency and specifies the particular record desired and the law enforcement activity for which the record is sought; • For compelling circumstances affecting the health and safety of any person if notice of the disclosure is sent to the last known address of the individual; • To either House of Congress or to any congressional committee or subcommittee (see s

In [51]:
results = vectorstore.similarity_search_with_score(
    question,
    k=10
)

for doc, score in results:
    print(score)
    print(doc.page_content[:500])

0.7919083833694458
of a beneficiary to the extent necessary to protect the beneficiary's rights under title II or XVIII. L. Disclosure without Consent Below are listed the situations in which data on identifiable individuals may be released without the individual's consent. The disclosure is permitted if the disclosure would be: • To DHHS employees and officers who need the records to perform their duties; • Required by the FOIA; • To the Bureau of Census; • For research purposes under certain circumstances; • To 
0.7920613288879395
(Rev. 1, 09-11-02) Sections 3764 - 3769 set the guidelines for disclosure of information about a named beneficiary, to whom it may be disclosed, and the purposes for which it may be disclosed. In general, no information may be released except to the beneficiary (or the beneficiary's legal guardian) without the beneficiary's (or legal guardian's) explicit written authorization. In cases requiring the beneficiary's consent, the authorization may be in any for